#### Faiss
Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [6]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("speech.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=30)
docs = text_splitter.split_documents(documents)


In [7]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct our

In [11]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

db=FAISS.from_documents(docs,embeddings)

db

In [12]:
### querying 
query="How does the speaker describe the desired outcome of the war?"
docs=db.similarity_search(query)
docs


[Document(id='8df6ddaa-cbc4-404c-bd72-cb021260ee63', metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='d1b6701f-63ab-4d28-83f0-828b92442cc1', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. Ther

In [13]:
docs[0].page_content

'…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

#### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [15]:
retriever=db.as_retriever()
docs=retriever.invoke(query)
docs

[Document(id='8df6ddaa-cbc4-404c-bd72-cb021260ee63', metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='d1b6701f-63ab-4d28-83f0-828b92442cc1', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. Ther

In [16]:
docs[0].page_content

'…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

#### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [17]:
docs_and_score=db.similarity_search_with_score(query)
docs_and_score

[(Document(id='8df6ddaa-cbc4-404c-bd72-cb021260ee63', metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
  np.float32(0.84652746)),
 (Document(id='d1b6701f-63ab-4d28-83f0-828b92442cc1', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed

In [18]:
embedding_vector=embeddings.embed_query(query)
embedding_vector

[-0.01800055,
 0.0063137733,
 -0.15973036,
 -0.04520984,
 0.100985155,
 0.07281908,
 -0.045555048,
 0.006389273,
 0.017462429,
 -0.00094360305,
 -0.008605399,
 0.029702188,
 0.044380285,
 0.06829806,
 0.094869696,
 -0.043996107,
 0.014207259,
 -0.045686137,
 -0.03568018,
 0.021441344,
 -0.02951707,
 -0.012393565,
 4.4729706e-05,
 0.031182826,
 0.07162357,
 0.024630899,
 -0.020619253,
 0.039723355,
 -0.03855573,
 -0.036176383,
 0.045613766,
 0.0048357043,
 -0.006432861,
 0.0043425234,
 -0.07232784,
 -0.07703563,
 0.0037108092,
 0.0622428,
 0.013771524,
 -0.05991044,
 -0.0060248026,
 -0.026603775,
 0.0018875591,
 -0.077041954,
 0.05505568,
 -0.020076163,
 -0.018464917,
 0.0060031153,
 -3.3080003e-05,
 -0.027851075,
 -0.035667695,
 -0.0295903,
 0.00487952,
 -0.010872308,
 0.05713685,
 0.0062902127,
 -0.033545244,
 -0.034675222,
 0.031607315,
 0.0029171503,
 0.024637314,
 0.010269885,
 -0.02696355,
 0.03752527,
 0.059708495,
 -0.03158323,
 -0.060380463,
 0.029953655,
 0.025182758,
 -0.0591

In [19]:
docs=db.similarity_search_by_vector(embedding_vector)
docs

[Document(id='8df6ddaa-cbc4-404c-bd72-cb021260ee63', metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='d1b6701f-63ab-4d28-83f0-828b92442cc1', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. Ther

In [20]:
### Saving And Loading
db.save_local("faiss_index")

In [21]:
new_db=FAISS.load_local("faiss_index",embeddings,allow_dangerous_deserialization=True)
docs=new_db.similarity_search(query)

In [22]:
docs

[Document(id='8df6ddaa-cbc4-404c-bd72-cb021260ee63', metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='d1b6701f-63ab-4d28-83f0-828b92442cc1', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. Ther